In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import chi2, spearmanr
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import joblib

In [2]:
data = pd.read_csv('Dataset/Final Dataset.csv')
data.head()

,Unnamed: 0,EnergyRating,BerRating,CO2Rating,MainSpaceHeatingFuel,CountyName,DwellingTypeDescr,Year_of_Construction,NoStoreys,GroundFloorArea(sq m),...,VentilationMethod,DraftLobby,PercentageDraughtStripped,NoOfSidesSheltered,SuspendedWoodenFloor,InsulationType,InsulationThickness,WaterStorageVolume,SolarHotWaterHeating,SolarPVEnergyOutput
0,205462,C3,212.41,52.29,Heating Oil,Co. Cork,Semi-detached house,1972,2,110.360,...,Natural vent.,YES,100.0,2.0,No,Loose Jacket,30.0,157.0,NO,0.00
1,1090514,B3,135.62,39.65,Heating Oil,Co. Cork,Detached house,1973,2,297.680,...,Natural vent.,NO,100.0,3.0,No,Factory Insulated,25.0,162.0,NO,6462.00
2,1090515,B2,108.33,23.09,Mains Gas,Co. Dublin,Semi-detached house,1999,2,105.820,...,Natural vent.,NO,100.0,2.0,No,Factory Insulated,25.0,117.0,NO,2107.34
3,1090517,D1,225.34,53.23,Heating Oil,Co. Longford,Detached house,1986,1,76.320,...,Natural vent.,NO,100.0,1.0,No,Factory Insulated,35.0,117.0,NO,0.00
4,1090520,C1,153.36,36.86,Mains Gas,Co. Dublin,End of terrace house,1985,2,77.652,...,Natural vent.,NO,93.0,2.0,No,Factory Insulated,35.0,88.0,NO,3924.10


In [3]:
print(data.columns)
print(data.shape)
SEED = 25223721

Index(['Unnamed: 0', 'EnergyRating', 'BerRating', 'CO2Rating',
       'MainSpaceHeatingFuel', 'CountyName', 'DwellingTypeDescr',
       'Year_of_Construction', 'NoStoreys', 'GroundFloorArea(sq m)',
       'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor',
       'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea',
       'StructureType', 'PredominantRoofType', 'RoomInRoofArea',
       'LivingAreaPercent', 'MainWaterHeatingFuel', 'HSMainSystemEfficiency',
       'WHMainSystemEff', 'HSSupplHeatFraction', 'HeatSystemControlCat',
       'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues',
       'NoOfFansAndVents', 'VentilationMethod', 'DraftLobby',
       'PercentageDraughtStripped', 'NoOfSidesSheltered',
       'SuspendedWoodenFloor', 'InsulationType', 'InsulationThickness',
       'WaterStorageVolume', 'SolarHotWaterHeating', 'SolarPVEnergyOutput'],
      dtype='object')
(105832, 43)


Creating appropiate directories

In [4]:
dirs = ['Figures','Outputs','Models']
for d in dirs:
    os.makedirs(d,exist_ok = True)

In [5]:
data = data.drop(columns = ['Unnamed: 0'],errors = 'ignoore').reset_index(drop = True)
if os.path.exists('Dataser/train_idx.csv'):
    train_idx = pd.read_csv('Dataset/train_idx.csv').iloc[:,0].values
    test_idx = pd.read_csv('Dataset/ttest_idx.csv').iloc[:,0].values
else:
    data = data.dropna(subset = ['EnergyRating']) # Needed to ensuee the split happens correctly
    train_idx,test_idx = train_test_split(data.index.values,test_size = 0.2,random_state = SEED,stratify = data['EnergyRating'])
    pd.Series(train_idx).to_csv('Dataset/train_idx.csv',index = False)
    pd.Series(test_idx).to_csv('Dataset/test_idx.csv',index = False)

## PCA

In [6]:
# We keep only continuous values since PXCA only works on that
exclude = ['BerRating','CO2Rating','HeatSystemControlCat','HeatResponseResponseCat']
num_col = [c for c in data.select_dtypes("number").columns if c not in exclude and data[c].nunique () > 2]
x_train,x_test = data.loc[train_idx,num_col], data.loc[test_idx,num_col]
print("Numeric Columns Chossen: ",num_col)

Numeric Columns Chossen:  ['Year_of_Construction', 'NoStoreys', 'GroundFloorArea(sq m)', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'RoomInRoofArea', 'LivingAreaPercent', 'HSMainSystemEfficiency', 'WHMainSystemEff', 'HSSupplHeatFraction', 'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'PercentageDraughtStripped', 'NoOfSidesSheltered', 'InsulationThickness', 'WaterStorageVolume', 'SolarPVEnergyOutput']


Data Preperation (Training data only)

In [7]:
skew = x_train.skew()
skewed = [ c for c in num_col if skew[c] >1 and x_train[c].min() >=0]
print('Skewed Columns: ',skewed)

Skewed Columns:  ['GroundFloorArea(sq m)', 'UValueWall', 'UValueRoof', 'WallArea', 'RoofArea', 'FloorArea', 'WindowArea', 'DoorArea', 'RoomInRoofArea', 'LivingAreaPercent', 'HSMainSystemEfficiency', 'WHMainSystemEff', 'HeatSystemResponseCat', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents', 'WaterStorageVolume', 'SolarPVEnergyOutput']


In [8]:
# Applying log transformation to all the skewed columns
def log_transform(df):
    df = df.copy()
    df[skewed] = np.log1p(df[skewed])
    return df

prep = Pipeline([
    ("imp", SimpleImputer(strategy='median')),
    ("sc", StandardScaler())
])
z_train = prep.fit_transform(log_transform(x_train))
z_test = prep.transform(log_transform(x_test))
assert z_train.shape[1] == len(num_col)


Correlation checking

In [10]:
R = np.corrcoef(z_train, rowvar=False)
n, p = z_train.shape
logdet = np.log(np.clip(np.linalg.eigvalsh(R), 1e-12, None)).sum()   # robust when columns are near-duplicates
bart = -(n - 1 - (2 * p + 5) / 6) * logdet
print(f"Bartlett chi2 = {bart:,.0f}, p-value = {chi2.sf(bart, p * (p - 1) / 2):.4g}")

inv = np.linalg.pinv(R)
partial = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
off = ~np.eye(p, dtype=bool)
kmo = (R[off] ** 2).sum() / ((R[off] ** 2).sum() + (partial[off] ** 2).sum())
print(f"KMO = {kmo:.3f}  (above 0.6 means PCA is suitable)")

plt.figure(figsize=(9, 7.5))
plt.imshow(R, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(p), num_col, rotation=90, fontsize=7); plt.yticks(range(p), num_col, fontsize=7)
plt.colorbar(label="Correlation"); plt.title("Correlation matrix (prepared training data)")
plt.tight_layout(); plt.savefig("pca_correlation.png", dpi=200); plt.close()

Bartlett chi2 = 1,164,417, p-value = 0
KMO = 0.868  (above 0.6 means PCA is suitable)


Fitting PCA

In [16]:
pca = PCA(random_state = SEED).fit(z_train)
eig,evr = pca.explained_variance_,pca.explained_variance_ratio_
cum = evr.cumsum()
k_kaiser = int( (eig>1).sum() )
k_90 = int( np.argmax( cum >= 0.90 ) +1)
k_95 = int( np.argmax( cum >= 0.95 ) +1)
print(f"Kaiser (eigenvalue > 1): {k_kaiser} | 90% variance: {k_90} | 95% variance: {k_95}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(range(1, p + 1), eig, "o-"); ax[0].axhline(1, ls="--", c="grey")
ax[0].set(xlabel="Component", ylabel="Eigenvalue", title="Scree plot")
ax[1].plot(range(1, p + 1), cum, "o-")
ax[1].axhline(0.90, ls="--", c="grey"); ax[1].axhline(0.95, ls=":", c="grey")
ax[1].set(xlabel="Number of components", ylabel="Cumulative explained variance", title="Cumulative variance")
plt.tight_layout(); plt.savefig("pca_scree.png", dpi=200); plt.close()

K = k_90 #By looking at the plot

Kaiser (eigenvalue > 1): 7 | 90% variance: 17 | 95% variance: 21


Feature Importance from PCA

In [21]:
pcs = [f"PC{i + 1}" for i in range(p)]
loadings = pd.DataFrame(pca.components_.T, index=num_col, columns=pcs)
for i, pc in enumerate(pcs[:4]):
    top = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index).head(6)
    print(f"\n{pc} ({evr[i]:.1%} of variance)\n{top.round(2).to_string()}")

importance = (((loadings.iloc[:, :K] ** 2) * evr[:K]).sum(axis=1) / evr[:K].sum()).sort_values(ascending=False)
print("\nFeature importance (sums to 1):\n", importance.round(3).to_string())

plt.figure(figsize=(8, 0.35 * p + 1))
plt.imshow(loadings.iloc[:, :K], cmap="coolwarm", vmin=-0.6, vmax=0.6, aspect="auto")
plt.xticks(range(K), pcs[:K]); plt.yticks(range(p), num_col, fontsize=7)
plt.colorbar(label="Loading"); plt.title(f"Loadings of the first {K} components")
plt.tight_layout(); plt.savefig("pca_loadings_heatmap.png", dpi=200); plt.close()

importance.sort_values().plot.barh(figsize=(7, 0.3 * p + 1), title=f"Feature importance (first {K} PCs)")
plt.xlabel("Variance-weighted squared loading"); plt.tight_layout()
plt.savefig("pca_importance.png", dpi=200); plt.close()


PC1 (26.1% of variance)
WallArea           0.26
UvalueDoor         0.26
UValueFloor        0.25
RoofArea           0.25
WHMainSystemEff   -0.25
FloorArea          0.25

PC2 (15.9% of variance)
GroundFloorArea(sq m)     0.31
HSMainSystemEfficiency    0.29
WindowArea                0.28
WHMainSystemEff           0.27
UValueWall               -0.27
WallArea                  0.27

PC3 (6.0% of variance)
UValueRoof              0.38
InsulationThickness    -0.30
NoStoreys               0.28
Year_of_Construction   -0.27
HSSupplHeatFraction    -0.27
NoOfFansAndVents       -0.26

PC4 (4.7% of variance)
NoStoreys             0.47
RoomInRoofArea        0.44
NoOfFansAndVents      0.40
NoOfSidesSheltered    0.31
NoOfOpenFlues        -0.25
DoorArea             -0.21

Feature importance (sums to 1):
 PercentageDraughtStripped    0.041
RoomInRoofArea               0.041
NoOfFansAndVents             0.041
SolarPVEnergyOutput          0.041
NoOfSidesSheltered           0.041
HeatSystemResponseCat      

In [24]:
def recon_share(Z, k):
    S = pca.transform(Z)
    rec = S[:, :k] @ pca.components_[:k] + pca.mean_
    return 1 - ((Z - rec) ** 2).sum() / ((Z - pca.mean_) ** 2).sum()
print(f"\nVariance retained with {K} PCs -> train: {recon_share(z_train, K):.3f} | test: {recon_share(z_test, K):.3f}")


rng = np.random.default_rng(SEED)
sims, rhos = [], []
for _ in range(5):
    sub = rng.choice(len(z_train), len(z_train) // 2, replace=False)
    ps = PCA(random_state=SEED).fit(z_train[sub])
    sims.append([abs(np.dot(ps.components_[i], pca.components_[i])) for i in range(3)])
    imp_s = pd.Series(((ps.components_[:K].T ** 2) * ps.explained_variance_ratio_[:K]).sum(axis=1)
                      / ps.explained_variance_ratio_[:K].sum(), index=num_col)
    rhos.append(spearmanr(imp_s[importance.index], importance.values)[0])
print("Stability (|cosine| of PC1-PC3 vs full fit):", np.round(np.mean(sims, axis=0), 3))
print("Stability (Spearman of importance ranking):", round(float(np.mean(rhos)), 3))


Variance retained with 17 PCs -> train: 0.907 | test: 0.907
Stability (|cosine| of PC1-PC3 vs full fit): [1. 1. 1.]
Stability (Spearman of importance ranking): 0.975


In [25]:
cum_imp = importance.cumsum()
n90 = int((cum_imp < 0.90).sum() + 1)
print(f"{n90} of {p} columns account for 90% of the importance:", list(importance.index[:n90]))

24 of 27 columns account for 90% of the importance: ['PercentageDraughtStripped', 'RoomInRoofArea', 'NoOfFansAndVents', 'SolarPVEnergyOutput', 'NoOfSidesSheltered', 'HeatSystemResponseCat', 'InsulationThickness', 'WaterStorageVolume', 'GroundFloorArea(sq m)', 'NoStoreys', 'UValueRoof', 'NoOfChimneys', 'DoorArea', 'HSSupplHeatFraction', 'WindowArea', 'WHMainSystemEff', 'Year_of_Construction', 'NoOfOpenFlues', 'UValueFloor', 'HSMainSystemEfficiency', 'FloorArea', 'UvalueDoor', 'UValueWindow', 'WallArea']


In [27]:
S_tr = pca.transform(z_train)
s = rng.choice(len(S_tr), min(10000, len(S_tr)), replace=False)
band = data.loc[train_idx, "EnergyRating"].astype(str).str[0].values[s]
for b in sorted(set(band)):
    m = band == b
    plt.scatter(S_tr[s][m, 0], S_tr[s][m, 1], s=4, alpha=0.4, label=b)
plt.legend(title="BER band", markerscale=3)
plt.xlabel(f"PC1 ({evr[0]:.1%})"); plt.ylabel(f"PC2 ({evr[1]:.1%})")
plt.tight_layout(); plt.savefig("pca_pc1_pc2.png", dpi=200); plt.close()

In [29]:
loadings.to_csv("Outputs/pca_loadings.csv")
importance.to_csv("Outputs/pca_feature_importance.csv")
joblib.dump({"prep": prep, "pca": pca, "skewed": skewed, "num_cols": num_col, "K": K},
            "Models/pca_pipeline.joblib")
pd.DataFrame(S_tr[:, :K], index=train_idx, columns=pcs[:K]).to_csv("Outputs/pca_scores_train.csv")
pd.DataFrame(pca.transform(z_test)[:, :K], index=test_idx, columns=pcs[:K]).to_csv("Outputs/pca_scores_test.csv")